In [ ]:
import numpy as np
import torch
from transformers import BertTokenizer, BertModel

# ---------------------------
# 1. 加载基因列表
# ---------------------------
gene_names = np.load('select_genes/cSCC_Selected_Genes.npy')
# 确保它是字符串列表
gene_list = gene_names.tolist()

# ---------------------------
# 2. 使用 BERT 进行编码
# ---------------------------
def get_bert_embeddings(texts, model_name='bert-base-uncased'):
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
            outputs = model(**inputs)
            # 取 [CLS] token 的向量作为整个词的表征 (768维)
            cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()
            embeddings.append(cls_embedding)
    return np.vstack(embeddings)

# ---------------------------
# 3. 使用 MedCLIP 进行编码
# ---------------------------
# 注意：MedCLIP 专门优化了医学术语的对齐
def get_medclip_embeddings(texts):
    from medclip import MedCLIPModel, MedCLIPProcessor
    
    # 初始化 MedCLIP (假设使用官方预训练权重)
    model = MedCLIPModel.from_pretrained("microsoft/MedCLIP-ViT-P16")
    processor = MedCLIPProcessor.from_pretrained("microsoft/MedCLIP-ViT-P16")
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for text in texts:
            # 编码文本
            inputs = processor(text=[text], return_tensors="pt", padding=True)
            text_embeds = model.encode_text(inputs['input_ids'], inputs['attention_mask'])
            embeddings.append(text_embeds.numpy())
    return np.vstack(embeddings)

# 执行编码
print("正在提取 BERT 特征...")
bert_features = get_bert_embeddings(gene_list)

print("正在提取 MedCLIP 特征...")
# 如果环境已配置好 MedCLIP，取消下行注释
# medclip_features = get_medclip_embeddings(gene_list)

# ---------------------------
# 4. 保存结果
# ---------------------------
np.save('gene_bert_embeddings.npy', bert_features)
print(f"BERT 特征提取完成，形状为: {bert_features.shape}")